# 🛒 สถิติเรื่องร้องทุกข์ผู้บริโภค กทม. (สคบ.) — ปี 2569

วิเคราะห์เรื่องร้องทุกข์ของผู้บริโภคในเขตกรุงเทพมหานคร จากสำนักงานคณะกรรมการคุ้มครองผู้บริโภค (สคบ.)  
ที่มา: public dashboard `ocpbconnect.ocpb.go.th` (สแนปช็อตปีปัจจุบัน 2569 · 50 เขต)

- **Treemap** — กลุ่มเขต → เขต (จำนวนเรื่องร้องทุกข์)
- **Composition** — สัดส่วนประเภทเรื่องร้องทุกข์แต่ละกลุ่มเขต + สถานะเรื่อง
- **K-Means Clustering** — จัดกลุ่มเขตตามโปรไฟล์เรื่องร้องทุกข์

> หมายเหตุ: dashboard สาธารณะแสดงข้อมูลปีปัจจุบันและไม่เปิดเผยชื่อประเภท 4 หมวด (แสดงเป็น หมวด 1–4) — ข้อมูลย้อนหลัง/ระดับแขวงต้องใช้ Web Service API (apiKey+password)

In [ ]:
# Google Colab — run this cell first to install dependencies
!pip install pandas plotly scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.io import to_html
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import os

df = pd.read_csv('ocpb_complaint_bkk.csv', encoding='utf-8-sig')
df.columns = df.columns.str.strip()
cat_cols = ['cat1','cat2','cat3','cat4']

STATUS = {'กำลังดำเนินการ':3261,'ยุติ':5200}
TOTAL_ALL = 8461

n_dist   = df['district'].nunique()
n_zone   = df['zone'].nunique()
tot_dist = int(df['total'].sum())
top_d    = df.loc[df['total'].idxmax(),'district']
top_v    = int(df['total'].max())

print(f'Districts : {n_dist}   Zones: {n_zone}')
print(f'Complaints: {tot_dist:,} (dashboard total {TOTAL_ALL:,})')
print(f'สถานะ     : ยุติ {STATUS["ยุติ"]:,} | กำลังดำเนินการ {STATUS["กำลังดำเนินการ"]:,}')
print(f'สูงสุด    : {top_d} ({top_v:,})')
df.head(3)

In [ ]:
PALETTE = [
    '#087e8b','#0b3954','#ff5a5f','#f6ae2d','#3aa17e',
    '#1b98a2','#136f63','#e8871e','#c5283d','#255f85',
    '#06d6a0','#118ab2','#ffd166','#8d99ae','#adb5bd',
]
CLUSTER_COLORS = ['#087e8b','#ff5a5f','#f6ae2d','#0b3954','#3aa17e']
CAT_NAMES = {'cat1':'หมวด 1','cat2':'หมวด 2','cat3':'หมวด 3','cat4':'หมวด 4'}
CAT_COLORS = ['#ff5a5f','#f6ae2d','#f9e79f','#7fb3d5']
ZONE_SHORT = {'กรุงเทพกลาง':'กทม.กลาง','กรุงเทพใต้':'กทม.ใต้','กรุงเทพเหนือ':'กทม.เหนือ',
              'กรุงเทพตะวันออก':'กทม.ตอ.','ธนบุรีเหนือ':'ธนเหนือ','ธนบุรีใต้':'ธนใต้'}
print('Palette ready.')

## Chart 1 — Treemap: กลุ่มเขต → เขต

ขนาด = จำนวนเรื่องร้องทุกข์ · คลิกเพื่อ zoom

In [ ]:
zone_tot = df.groupby('zone')['total'].sum().sort_values(ascending=False)
ids=['กทม.']; labels=['กรุงเทพฯ']; parents=['']; values=[int(df['total'].sum())]; mcolors=['rgba(0,0,0,0)']
cmap={}
for i,(z,t) in enumerate(zone_tot.items()):
    c=PALETTE[i%len(PALETTE)]; cmap[z]=c
    ids.append(z); labels.append(ZONE_SHORT.get(z,z)); parents.append('กทม.')
    values.append(int(t)); mcolors.append(c)
for _,r in df.sort_values('total',ascending=False).iterrows():
    ids.append(f"{r['zone']}|{r['district']}"); labels.append(r['district']); parents.append(r['zone'])
    values.append(int(r['total'])); mcolors.append(cmap.get(r['zone'],'#ccc')+'cc')
fig_tree = go.Figure(go.Treemap(
    ids=ids, labels=labels, parents=parents, values=values, branchvalues='total',
    marker=dict(colors=mcolors, line=dict(color='white',width=2), pad=dict(t=26,l=4,r=4,b=4)),
    texttemplate='<b>%{label}</b><br>%{value:,}', textposition='middle center',
    textfont=dict(size=13,color='white'),
    hovertemplate='<b>%{label}</b><br>เรื่องร้องทุกข์: %{value:,}<br>%{percentRoot:.1%} ของ กทม.<extra></extra>',
    maxdepth=2))
fig_tree.update_layout(title='เรื่องร้องทุกข์ผู้บริโภค: กลุ่มเขต → เขต (คลิกเพื่อ zoom)',
    height=560, margin=dict(t=50,l=4,r=4,b=4), uniformtext=dict(minsize=10,mode='hide'))
fig_tree.show()

## Chart 2 — แผนที่กรุงเทพฯ: เรื่องร้องทุกข์ต่อประชากร

Choropleth รายเขต · สี = เรื่องร้องทุกข์ต่อประชากร 10,000 คน (ประชากรจาก geojson สำมะโน)

In [ ]:
import json, urllib.request
def _load_geojson(local, url):
    try:    return json.load(open(local, encoding='utf-8'))
    except Exception:
        with urllib.request.urlopen(url, timeout=30) as r: return json.load(r)
bkk_geo = _load_geojson('bkk_districts.geojson',
    'https://raw.githubusercontent.com/pcrete/gsvloader-demo/master/geojson/Bangkok-districts.geojson')
pop = {}
for f in bkk_geo['features']:
    p = f['properties']; nm = p['dname'].replace('เขต','',1).strip().replace('ราษฏร์บูรณะ','ราษฎร์บูรณะ')
    p['dname'] = 'เขต'+nm   # unify key for featureidkey
    pop[nm] = (p.get('no_male') or 0)+(p.get('no_female') or 0)
df['pop']    = df['district'].map(pop)
df['per10k'] = (df['total']/df['pop']*10000).round(2)
df['dkey']   = 'เขต'+df['district']
fig_map = px.choropleth(df, geojson=bkk_geo, featureidkey='properties.dname',
    locations='dkey', color='per10k', color_continuous_scale='Teal',
    custom_data=['district','total','pop','per10k'])
fig_map.update_traces(hovertemplate=('<b>%{customdata[0]}</b><br>เรื่องร้องทุกข์: %{customdata[1]:,}<br>'
    'ประชากร: %{customdata[2]:,}<br>ต่อ 10,000 คน: %{customdata[3]}<extra></extra>'))
fig_map.update_geos(fitbounds='locations', visible=False)
fig_map.update_layout(title='เรื่องร้องทุกข์ต่อประชากร 10,000 คน รายเขต (ปี 2569)',
    height=620, margin=dict(t=50,l=0,r=0,b=0), coloraxis_colorbar=dict(title='ต่อ 10k'))
fig_map.show()
print('ต่อประชากรสูงสุด:', df.nlargest(3,'per10k')[['district','total','pop','per10k']].to_dict('records'))

## Chart 3 — สัดส่วนประเภทเรื่องร้องทุกข์ แยกตามกลุ่มเขต

(สแนปช็อตปีเดียว จึงแสดงองค์ประกอบประเภทแทน time series)

In [ ]:
zc = df.groupby('zone')[cat_cols].sum()
zc = zc.loc[zc.sum(axis=1).sort_values(ascending=False).index]
zc_norm = zc.div(zc.sum(axis=1), axis=0)
fig_ts = go.Figure()
for i,cc in enumerate(cat_cols):
    fig_ts.add_trace(go.Bar(
        name=CAT_NAMES[cc], y=[ZONE_SHORT.get(z,z) for z in zc_norm.index], x=zc_norm[cc].values,
        orientation='h', marker_color=CAT_COLORS[i],
        hovertemplate=f'<b>{CAT_NAMES[cc]}</b><br>สัดส่วน: %{{x:.1%}}<extra></extra>'))
fig_ts.update_layout(barmode='stack',
    title='สัดส่วนประเภทเรื่องร้องทุกข์ (4 หมวด) แยกตามกลุ่มเขต',
    xaxis_title='สัดส่วน', yaxis_title='กลุ่มเขต', xaxis=dict(tickformat='.0%'),
    height=430, plot_bgcolor='white', legend=dict(title='<b>ประเภท</b>', orientation='h', y=-0.18),
    margin=dict(t=55,b=70,l=90,r=20))
fig_ts.update_xaxes(showgrid=True, gridcolor='#e9ecef')
fig_ts.show()

## Chart 4 — Top 15 เขต + สถานะเรื่อง

In [ ]:
top = df.sort_values('total',ascending=False).head(15)
fig_bar = go.Figure(go.Bar(
    x=top['total'][::-1].values, y=top['district'][::-1].values, orientation='h',
    marker=dict(color=top['total'][::-1].values, colorscale='Teal', line=dict(color='white',width=1)),
    text=top['total'][::-1].apply(lambda v:f'{v:,}'), textposition='outside',
    hovertemplate='<b>%{y}</b><br>เรื่องร้องทุกข์: %{x:,}<extra></extra>'))
fig_bar.update_layout(title='15 เขตที่มีเรื่องร้องทุกข์มากที่สุด (ปี 2569)',
    xaxis_title='จำนวนเรื่อง', height=470, plot_bgcolor='white', margin=dict(t=55,b=50,l=110,r=70))
fig_bar.update_xaxes(showgrid=True, gridcolor='#e9ecef')
fig_bar.show()

fig_donut = go.Figure(go.Pie(
    labels=list(STATUS.keys()), values=list(STATUS.values()), hole=.58,
    marker=dict(colors=['#f6ae2d','#087e8b'], line=dict(color='white',width=2)),
    textinfo='label+percent', textfont=dict(size=14)))
fig_donut.update_layout(title='สถานะเรื่องร้องทุกข์ทั้งหมด (ปี 2569)',
    height=360, margin=dict(t=55,b=20,l=20,r=20),
    annotations=[dict(text=f'{TOTAL_ALL:,}<br>เรื่อง', x=.5, y=.5, font_size=18, showarrow=False)])
fig_donut.show()

## K-Means Clustering — จัดกลุ่มเขตตามโปรไฟล์เรื่องร้องทุกข์

**Feature ที่ใช้ (7 features)** ต่อเขต:
- **สัดส่วนประเภท (4)** — % ของ หมวด 1 / 2 / 3 / 4
- **log_volume** — ปริมาณเรื่องร้องทุกข์ (log)
- **top_share** — สัดส่วนของหมวดที่มากที่สุด (ความกระจุกตัว)
- **entropy** — ความหลากหลายของประเภท (Shannon entropy)

In [ ]:
feat = pd.DataFrame(index=df['district'])
tot = df.set_index('district')['total']
cc  = df.set_index('district')[cat_cols]
prop = cc.div(cc.sum(axis=1).replace(0,np.nan), axis=0).fillna(0)
for c in cat_cols: feat['p_'+c] = prop[c].values
feat['log_vol']   = np.log1p(tot.values)
feat['top_share'] = prop.max(axis=1).values
def _ent(row):
    p=row[row>0].values; return float(-np.sum(p*np.log(p+1e-12)))
feat['entropy']   = prop.apply(_ent, axis=1).values

prop_cols=['p_'+c for c in cat_cols]
X = StandardScaler().fit_transform(feat.values)
k = min(5, max(3, feat.shape[0]//12))
km = KMeans(n_clusters=k, random_state=42, n_init=10)
clab = km.fit_predict(X)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X); var_exp = pca.explained_variance_ratio_

zmap = df.set_index('district')['zone'].to_dict()
pca_df = pd.DataFrame({'PC1':coords[:,0],'PC2':coords[:,1],'district':feat.index,
    'cluster':clab.astype(str),'total':tot.reindex(feat.index).values,
    'top_share':feat['top_share'].values,'zone':[zmap.get(d,'') for d in feat.index]})
feat_c=feat.copy(); feat_c['cluster']=clab
prop_means = feat_c.groupby('cluster')[prop_cols].mean()
meta = feat_c.groupby('cluster')[['log_vol','top_share','entropy']].mean()
meta['n']=feat_c.groupby('cluster').size()
print(f'Feature matrix: {feat.shape[0]} districts x {feat.shape[1]} features')
print(f'k = {k}   PC1={var_exp[0]:.1%} PC2={var_exp[1]:.1%}')
for c in sorted(pca_df['cluster'].unique(),key=int):
    s=pca_df[pca_df['cluster']==c]
    print(f"  กลุ่ม {c}: {len(s):2d} เขต  {int(s['total'].sum()):>5,} เรื่อง")

### Chart 4 — PCA Scatter: K-Means กลุ่มเขต

ขนาดจุด = จำนวนเรื่อง · สี = กลุ่ม

In [ ]:
fig_pca = go.Figure()
for c in sorted(pca_df['cluster'].unique(),key=int):
    sub=pca_df[pca_df['cluster']==c]
    fig_pca.add_trace(go.Scatter(x=sub['PC1'],y=sub['PC2'],mode='markers+text',name=f'กลุ่ม {c}',
        marker=dict(size=sub['total'].apply(lambda x:max(12,min(48,np.log1p(x)*6))),
                    color=CLUSTER_COLORS[int(c)%len(CLUSTER_COLORS)],opacity=.85,line=dict(color='white',width=1.5)),
        text=sub['district'],textposition='top center',textfont=dict(size=9),
        customdata=sub[['district','total','zone','top_share']].values,
        hovertemplate=('<b>%{customdata[0]}</b><br>เรื่อง: %{customdata[1]:,}<br>'
                       'กลุ่มเขต: %{customdata[2]}<br>หมวดเด่น: %{customdata[3]:.0%}<extra></extra>')))
fig_pca.update_layout(title=(f'K-Means (k={k}) — PCA Projection | 7 features<br>'
    f'<sup>PC1={var_exp[0]:.1%}, PC2={var_exp[1]:.1%} ของความแปรปรวน</sup>'),
    xaxis_title=f'PC1 ({var_exp[0]:.1%})', yaxis_title=f'PC2 ({var_exp[1]:.1%})',
    height=520, plot_bgcolor='white', legend=dict(title='<b>กลุ่ม K-Means</b>'), margin=dict(t=75,b=50,l=65,r=20))
fig_pca.update_xaxes(showgrid=True,gridcolor='#e9ecef',zeroline=True,zerolinecolor='#dee2e6')
fig_pca.update_yaxes(showgrid=True,gridcolor='#e9ecef',zeroline=True,zerolinecolor='#dee2e6')
fig_pca.show()

### Chart 5 — Cluster Profiles: แต่ละกลุ่มร้องเรื่องหมวดไหน?

In [ ]:
fig_prof = go.Figure()
for i,pc in enumerate(prop_cols):
    nm=CAT_NAMES[pc.replace('p_','')]
    fig_prof.add_trace(go.Bar(name=nm, x=[f'กลุ่ม {c}' for c in prop_means.index], y=prop_means[pc].values,
        marker_color=CAT_COLORS[i], hovertemplate=f'<b>{nm}</b><br>สัดส่วน: %{{y:.1%}}<extra></extra>'))
fig_prof.update_layout(barmode='stack', title='โปรไฟล์ประเภทเรื่องร้องทุกข์ของแต่ละกลุ่ม K-Means',
    xaxis_title='กลุ่ม K-Means', yaxis_title='สัดส่วนเฉลี่ย', yaxis=dict(tickformat='.0%'),
    height=470, plot_bgcolor='white', legend=dict(title='ประเภท', font_size=11, x=1.01, y=.99),
    margin=dict(t=55,b=50,l=70,r=140))
fig_prof.update_yaxes(showgrid=True,gridcolor='#e9ecef'); fig_prof.update_xaxes(showgrid=False)
fig_prof.show()
print('\n── ปริมาณ & ความกระจุกตัว เฉลี่ยต่อกลุ่ม ──')
print(meta.rename(columns={'log_vol':'log ปริมาณ','top_share':'หมวดเด่น','entropy':'ความหลากหลาย','n':'จำนวนเขต'}).round(3).to_string())

## 🔮 Predictive Model — ทำนายจำนวนเรื่องร้องทุกข์รายเขต (Ridge Regression)

**โมเดล:** Ridge Regression ทำนาย*จำนวนเรื่องร้องทุกข์*ของเขต จาก **ประชากร + สัดส่วนประเภท + กลุ่มเขต**  
ประเมินด้วย **5-fold cross-validation** → แสดง *predicted vs actual* พร้อม R² และ MAE  
(ประชากรมีความสัมพันธ์กับจำนวนเรื่องร้องทุกข์ ~0.68 — เป็น feature หลัก)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_absolute_error
reg = df.copy()
reg['catsum'] = reg[cat_cols].sum(axis=1)
for c in cat_cols: reg['pp_'+c] = reg[c]/reg['catsum']
Zd = pd.get_dummies(reg['zone'], prefix='z')
Xr = pd.concat([reg['pop']/1e5, reg[['pp_'+c for c in cat_cols]], Zd], axis=1).fillna(0)
yr = reg['total'].values
model = Ridge(alpha=1.0)
yhat = cross_val_predict(model, Xr, yr, cv=KFold(5, shuffle=True, random_state=42))
r2 = r2_score(yr, yhat); mae = mean_absolute_error(yr, yhat)
model.fit(Xr, yr)

lim = max(yr.max(), yhat.max())*1.05
fig_reg = go.Figure()
fig_reg.add_trace(go.Scatter(x=[0,lim], y=[0,lim], mode='lines', name='เส้นทำนายตรงจริง',
    line=dict(color='#adb5bd', dash='dash'), hoverinfo='skip'))
fig_reg.add_trace(go.Scatter(x=yr, y=yhat, mode='markers+text', name='เขต',
    marker=dict(size=11, color=reg['total'], colorscale='Teal', line=dict(color='white',width=1), showscale=False),
    text=reg['district'], textposition='top center', textfont=dict(size=8),
    customdata=np.stack([reg['district'], yr, yhat], axis=1),
    hovertemplate='<b>%{customdata[0]}</b><br>จริง: %{customdata[1]:,}<br>ทำนาย: %{customdata[2]:.0f}<extra></extra>'))
fig_reg.update_layout(title=f'Predicted vs Actual — Ridge Regression (5-fold CV) · R²={r2:.2f}, MAE={mae:.0f}',
    xaxis_title='จำนวนจริง (Actual)', yaxis_title='จำนวนที่ทำนาย (Predicted)',
    height=520, plot_bgcolor='white', showlegend=False, margin=dict(t=55,b=50,l=65,r=20))
fig_reg.update_xaxes(showgrid=True, gridcolor='#e9ecef')
fig_reg.update_yaxes(showgrid=True, gridcolor='#e9ecef')
fig_reg.show()
coef = sorted(zip(Xr.columns, model.coef_), key=lambda t:-abs(t[1]))[:6]
print(f'Ridge R²={r2:.2f}  MAE={mae:.0f}')
print('อิทธิพลของ feature (coef):')
for f,cf in coef: print(f'  {f:24} {cf:+.1f}')

## Export → ocpb_viz.html

In [ ]:
CSS = """
*{box-sizing:border-box}
body{font-family:'Sarabun','Segoe UI',sans-serif;background:#eef2f1;margin:0;padding:0;color:#212529}
.page{max-width:1140px;margin:0 auto;padding:32px 24px 64px}
.hero{background:linear-gradient(135deg,#04242b 0%,#0b3954 38%,#087e8b 72%,#3aa17e 100%);border-radius:16px;padding:40px 48px;margin-bottom:24px;color:white;position:relative;overflow:hidden}
.hero::before{content:'🛒';position:absolute;right:40px;top:14px;font-size:96px;opacity:.12;line-height:1}
.hero h1{margin:0 0 6px;font-size:2.0em;font-weight:700;letter-spacing:-.5px}
.hero p{margin:0;opacity:.78;font-size:1em}
.badge{display:inline-block;background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.25);border-radius:20px;padding:3px 13px;font-size:.83em;margin:12px 6px 0 0}
.stats{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:24px}
.stat-card{background:white;border-radius:12px;padding:20px 22px;box-shadow:0 1px 5px rgba(0,0,0,.08);border-left:4px solid var(--ac)}
.stat-card .num{font-size:1.9em;font-weight:700;color:var(--ac);line-height:1.1}
.stat-card .lbl{font-size:.84em;color:#6c757d;margin-top:5px}
.card{background:white;border-radius:12px;box-shadow:0 1px 5px rgba(0,0,0,.08);margin-bottom:24px;overflow:hidden}
.card-header{padding:16px 22px 0;font-size:.78em;font-weight:700;text-transform:uppercase;letter-spacing:.07em;color:#6c757d}
.grid-2{display:grid;grid-template-columns:1fr 1fr;gap:24px}
.note{font-size:.82em;color:#6c757d;background:#fff;border-left:4px solid #f6ae2d;border-radius:8px;padding:12px 18px;margin-bottom:24px;box-shadow:0 1px 5px rgba(0,0,0,.06)}
@media(max-width:768px){.stats{grid-template-columns:1fr 1fr}.grid-2{grid-template-columns:1fr}.hero h1{font-size:1.4em}.hero::before{display:none}}
@keyframes cartFall{0%{top:-60px;opacity:1}100%{top:110vh;opacity:0}}
@keyframes popIn{0%{opacity:0;transform:translate(-50%,-50%) scale(.05)}60%{transform:translate(-50%,-50%) scale(1.2)}80%{transform:translate(-50%,-50%) scale(.95)}100%{opacity:1;transform:translate(-50%,-50%) scale(1)}}
@keyframes fadeOut{to{opacity:0;transform:translate(-50%,-50%) scale(.6)}}
@keyframes glowPulse{0%,100%{box-shadow:0 0 40px rgba(8,126,139,.7),0 0 80px rgba(58,161,126,.4)}50%{box-shadow:0 0 80px rgba(8,126,139,1),0 0 160px rgba(58,161,126,.7)}}
@keyframes cartRun{0%{left:-90px}100%{left:110vw}}
@keyframes tealTint{0%{opacity:0}50%{opacity:.15}100%{opacity:0}}
"""

SHIELD_JS = r"""<script>
(function(){
  var busy=false;
  function mk(css,html){var e=document.createElement('div');e.style.cssText=css;if(html!==undefined)e.innerHTML=html;return e;}
  function go(){
    if(busy)return; busy=true;
    var trash=[]; function add(e){document.body.appendChild(e);trash.push(e);return e;}
    var ICONS=['🛒','⚖️','🛡️','🧾','📢','✅'];
    for(var i=0;i<32;i++){(function(idx){setTimeout(function(){
      add(mk('position:fixed;top:-60px;left:'+(2+idx*3.0)+'%;font-size:'+(16+Math.random()*16)+'px;'+
        'z-index:9980;pointer-events:none;animation:cartFall '+(1.6+Math.random()*2)+'s linear forwards;opacity:.85',
        ICONS[Math.floor(Math.random()*ICONS.length)]));},idx*60);})(i);}
    var msgs=['🧾 กำลังตรวจสอบเรื่องร้องทุกข์ 8,461 เรื่อง...','⚖️ พิจารณาตามกฎหมายคุ้มครองผู้บริโภค...','📢 แจ้งผู้ประกอบธุรกิจ...',
      '✅ ไกล่เกลี่ยสำเร็จ! ผู้บริโภคได้รับความเป็นธรรม','🛡️ คุ้มครองสิทธิผู้บริโภคทั่ว กทม.','💪 สคบ. ดูแลทุกเรื่องร้องทุกข์','🙏 ขอบคุณที่ร้องเรียน'];
    msgs.forEach(function(m,i){setTimeout(function(){
      var t=add(mk('position:fixed;top:'+(8+i*9)+'%;right:24px;z-index:9993;pointer-events:none;'+
        'background:linear-gradient(90deg,#0b3954,#087e8b);color:#fff;font-family:"Sarabun",sans-serif;'+
        'font-size:13px;padding:8px 18px;border-radius:20px;box-shadow:0 2px 12px rgba(8,126,139,.5)',m));
      t.style.transform='translateX(120%)';t.style.transition='transform .4s cubic-bezier(.17,.67,.35,1.3)';
      setTimeout(function(){t.style.transform='translateX(0)';},20);},400+i*350);});
    ['🛒','🛍️','🛒','🧺','🛒'].forEach(function(em,i){setTimeout(function(){
      add(mk('position:fixed;bottom:'+(6+i*7)+'%;left:-90px;font-size:'+(30+Math.random()*18)+'px;'+
        'z-index:9985;pointer-events:none;animation:cartRun '+(1.4+Math.random()*1.1)+'s linear forwards',em));},2200+i*260);});
    setTimeout(function(){add(mk('position:fixed;inset:0;background:#087e8b;z-index:9970;pointer-events:none;animation:tealTint 2s ease forwards'));},3200);
    setTimeout(function(){
      var box=add(mk('position:fixed;top:50%;left:50%;z-index:10000;pointer-events:none;text-align:center;'+
        'background:linear-gradient(135deg,#04242b 0%,#0b3954 50%,#087e8b 100%);color:white;font-size:26px;font-weight:900;'+
        'padding:32px 52px;border-radius:28px;animation:popIn .65s cubic-bezier(.17,.67,.35,1.3) forwards,glowPulse 1s ease .7s infinite;'+
        'font-family:"Sarabun",sans-serif;line-height:1.7;min-width:360px',
        '🛡️ คุ้มครองผู้บริโภคสำเร็จ! 🛡️<br><span style="font-size:17px">ดูแลเรื่องร้องทุกข์ 8,461 เรื่อง</span><br>'+
        '<span style="font-size:15px;opacity:.85">ผู้บริโภคยิ้มได้ทั่ว กทม. 😊</span><br>'+
        '<span style="font-size:12px;opacity:.6">สายด่วน สคบ. 1166 🙏</span>'));
      setTimeout(function(){box.style.animation='fadeOut .8s ease forwards';
        setTimeout(function(){trash.forEach(function(e){try{e.parentNode.removeChild(e);}catch(x){}});busy=false;},800);},3500);
    },4200);
  }
  window.addEventListener('load',function(){
    var btn=mk('position:fixed;bottom:28px;right:28px;font-size:46px;cursor:pointer;z-index:8000;'+
      'filter:drop-shadow(0 4px 12px rgba(8,126,139,.5));transition:transform .15s;user-select:none;line-height:1','🛒');
    btn.addEventListener('click',go);
    btn.addEventListener('mouseenter',function(){btn.style.transform='scale(1.25) rotate(-10deg)';});
    btn.addEventListener('mouseleave',function(){btn.style.transform='';});
    document.body.appendChild(btn);
    var tip=mk('position:fixed;bottom:84px;right:28px;background:#212529dd;color:white;font-size:12px;padding:5px 11px;'+
      'border-radius:8px;z-index:8000;opacity:0;transition:opacity .2s;pointer-events:none;white-space:nowrap;font-family:Sarabun,sans-serif','กดเพื่อพิทักษ์สิทธิ์! 🛡️');
    document.body.appendChild(tip);
    btn.addEventListener('mouseenter',function(){tip.style.opacity='1';});
    btn.addEventListener('mouseleave',function(){tip.style.opacity='0';});
  });
})();
</script>"""

html = f"""<!DOCTYPE html>
<html lang="th"><head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>สคบ. — เรื่องร้องทุกข์ผู้บริโภค กทม. 2569</title>
<link href="https://fonts.googleapis.com/css2?family=Sarabun:wght@400;600;700&display=swap" rel="stylesheet">
<style>{CSS}</style>
</head><body><div class="page">
<div class="hero">
  <h1>🛒 เรื่องร้องทุกข์ผู้บริโภค กทม. — สคบ. ปี 2569</h1>
  <p>วิเคราะห์เรื่องร้องทุกข์ผู้บริโภคในเขตกรุงเทพมหานคร · Python, Plotly &amp; Scikit-learn</p>
  <span class="badge">สำนักงานคณะกรรมการคุ้มครองผู้บริโภค (สคบ.)</span>
  <span class="badge">{TOTAL_ALL:,} เรื่อง</span>
  <span class="badge">{n_dist} เขต</span>
  <span class="badge">{n_zone} กลุ่มเขต</span>
  <span class="badge">🔮 Ridge R²={r2:.2f}</span>
</div>
<div class="stats">
  <div class="stat-card" style="--ac:#087e8b"><div class="num">{TOTAL_ALL:,}</div><div class="lbl">เรื่องร้องทุกข์ทั้งหมด</div></div>
  <div class="stat-card" style="--ac:#0b3954"><div class="num">{n_dist}</div><div class="lbl">เขต · {n_zone} กลุ่มเขต</div></div>
  <div class="stat-card" style="--ac:#3aa17e"><div class="num">{STATUS['ยุติ']/TOTAL_ALL:.0%}</div><div class="lbl">ยุติเรื่องแล้ว</div></div>
  <div class="stat-card" style="--ac:#ff5a5f"><div class="num">{top_d}</div><div class="lbl">เขตร้องทุกข์สูงสุด ({top_v:,})</div></div>
</div>
<div class="note">📊 ข้อมูลจาก public dashboard ของ สคบ. (ocpbconnect.ocpb.go.th) — สแนปช็อตปี 2569 · dashboard สาธารณะไม่เปิดเผยชื่อ 4 หมวดประเภท และไม่มีข้อมูลย้อนหลัง (ต้องใช้ Web Service API ที่ต้องมี apiKey+password)</div>
<div class="card"><div class="card-header">Treemap — กลุ่มเขต → เขต (คลิกเพื่อ zoom)</div>
  {to_html(fig_tree, full_html=False, include_plotlyjs=True)}</div>
<div class="card"><div class="card-header">🗺️ แผนที่ — เรื่องร้องทุกข์ต่อประชากร 10,000 คน</div>
  {to_html(fig_map, full_html=False, include_plotlyjs=False)}</div>
<div class="card"><div class="card-header">สัดส่วนประเภทเรื่องร้องทุกข์ แยกตามกลุ่มเขต</div>
  {to_html(fig_ts, full_html=False, include_plotlyjs=False)}</div>
<div class="grid-2">
  <div class="card"><div class="card-header">Top 15 เขต</div>
    {to_html(fig_bar, full_html=False, include_plotlyjs=False)}</div>
  <div class="card"><div class="card-header">สถานะเรื่อง</div>
    {to_html(fig_donut, full_html=False, include_plotlyjs=False)}</div>
</div>
<div class="grid-2">
  <div class="card"><div class="card-header">K-Means — PCA Projection</div>
    {to_html(fig_pca, full_html=False, include_plotlyjs=False)}</div>
  <div class="card"><div class="card-header">Cluster Profiles</div>
    {to_html(fig_prof, full_html=False, include_plotlyjs=False)}</div>
</div>
<div class="card"><div class="card-header">🔮 Predictive Model — Ridge Regression (Predicted vs Actual)</div>
  {to_html(fig_reg, full_html=False, include_plotlyjs=False)}</div>
</div>
{SHIELD_JS}
</body></html>"""

with open('ocpb_viz.html','w',encoding='utf-8') as f:
    f.write(html)
print(f'Saved: ocpb_viz.html ({os.path.getsize("ocpb_viz.html")//1024} KB)')